# 05 · Reportes automáticos y gráficos dinámicos

Genera salidas accionables a partir de `ira_resultados`:

- **Resumen ejecutivo nacional** (KPIs).
- **Ranking** de municipios en riesgo Alto/Crítico → exportable a CSV/HTML.
- **Gráficos** por departamento y cultivo.
- **Reporte por municipio** parametrizable — la versión estática de lo que el asistente
  LLM entrega en la app (`/reporte/{codigo}`).


In [ ]:
# Arranque: localizar la raiz del repo y la base de datos DuckDB
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import duckdb
import numpy as np
import pandas as pd

from config import config

DB_PATH = ROOT / config.duckdb_path
DB_OK = DB_PATH.exists()
print("BD:", DB_PATH, "->", "OK" if DB_OK else "NO existe (corre `make pipeline`)")


def con():
    if not DB_OK:
        raise FileNotFoundError(f"No existe {DB_PATH}. Ejecuta el pipeline primero.")
    c = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        c.execute("INSTALL spatial; LOAD spatial;")
    except Exception:
        pass
    return c


def q(sql: str) -> pd.DataFrame:
    with con() as c:
        return c.execute(sql).df()


In [ ]:
# Librerias de visualizacion
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    pass
plt.rcParams["figure.figsize"] = (10, 5)
NIVEL_COLOR = {"Bajo": "#2e7d32", "Medio": "#f9a825", "Alto": "#ef6c00", "Crítico": "#c62828"}


In [ ]:
# Nombres desde municipios_geom (no estan en ira_resultados)
ira = q("""
    SELECT r.*, g.nombre_municipio, g.nombre_departamento
    FROM ira_resultados r
    LEFT JOIN municipios_geom g USING (codigo_municipio)
""")

# Ultima foto por municipio+cultivo (periodo mas reciente)
ira_last = (ira.sort_values("periodo")
            .groupby(["codigo_municipio", "cultivo"], as_index=False).tail(1))
print("Registros (ultima foto):", len(ira_last))


## 1. Resumen ejecutivo nacional


In [ ]:
total = len(ira_last)
kpis = {
    "Municipios analizados": ira_last["codigo_municipio"].nunique(),
    "Cultivos analizados": ira_last["cultivo"].nunique(),
    "IRA medio nacional": round(ira_last["ira_score"].mean(), 3),
    "% en riesgo Alto/Critico": round(
        ira_last["ira_nivel"].isin(["Alto", "Crítico"]).mean() * 100, 1),
    "Anomalias detectadas": int(ira_last.get("is_anomaly", pd.Series(dtype=int)).sum()),
}
for k, v in kpis.items():
    print(f"{k:>28}: {v}")


In [ ]:
# Distribucion por nivel (grafico donut)
dist = ira_last["ira_nivel"].value_counts().reindex(
    ["Bajo", "Medio", "Alto", "Crítico"]).dropna()
colores = [NIVEL_COLOR.get(n, "#607d8b") for n in dist.index]

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(dist.values, labels=dist.index, colors=colores, autopct="%1.1f%%",
       wedgeprops=dict(width=0.42, edgecolor="white"), startangle=90)
ax.set_title("Distribucion nacional por nivel de riesgo")
plt.tight_layout(); plt.show()


## 2. Ranking de municipios en riesgo → exportable


In [ ]:
ranking = (ira_last[ira_last["ira_nivel"].isin(["Alto", "Crítico"])]
           .sort_values("ira_score", ascending=False)
           [["nombre_municipio", "nombre_departamento", "cultivo", "periodo",
             "ira_score", "ira_nivel", "spc", "sep", "sve", "rendimiento_predicho"]]
           .head(50).round(3))

# Exportar a CSV y HTML (reportes descargables)
rep_dir = ROOT / "notebooks" / "reportes"
rep_dir.mkdir(exist_ok=True)
ranking.to_csv(rep_dir / "ranking_riesgo.csv", index=False)
ranking.to_html(rep_dir / "ranking_riesgo.html", index=False)
print("Exportado a", rep_dir)
ranking.head(20)


## 3. Gráfico dinámico por dimensión


In [ ]:
def grafico_riesgo(dimension="nombre_departamento", top=15):
    """Grafico de barras del IRA medio por la dimension elegida."""
    g = (ira_last.groupby(dimension)
         .agg(ira_medio=("ira_score", "mean"), n=("ira_score", "size"))
         .query("n >= 10").sort_values("ira_medio", ascending=False).head(top))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(g.index[::-1], g["ira_medio"][::-1], color="#ef6c00", edgecolor="white")
    ax.set_title(f"IRA medio por {dimension} (top {top})")
    ax.set_xlabel("IRA medio")
    plt.tight_layout(); plt.show()
    return g.round(3)

# Cambia la dimension: "nombre_departamento" o "cultivo"
grafico_riesgo("nombre_departamento", top=15)


In [ ]:
grafico_riesgo("cultivo", top=15)


## 4. Reporte por municipio (parametrizable)

Versión estática del reporte ejecutivo que en la app genera el asistente LLM.
Cambia `CODIGO` por cualquier código DANE de 5 dígitos.


In [ ]:
def reporte_municipio(codigo: str):
    d = ira[ira["codigo_municipio"].astype(str) == str(codigo)]
    if d.empty:
        print(f"Sin datos para el municipio {codigo}")
        return
    ultimo = d.sort_values("periodo").iloc[-1]
    print("=" * 60)
    print("REPORTE DE RIESGO AGROCLIMATICO")
    print(f"Municipio : {ultimo['nombre_municipio']} ({ultimo['nombre_departamento']})")
    print(f"Codigo    : {codigo}   Periodo: {ultimo['periodo']}")
    print("=" * 60)
    print(f"IRA        : {ultimo['ira_score']:.3f}  ->  {ultimo['ira_nivel']}")
    print(f"  SPC (clima)          0.5 * {ultimo['spc']:.3f}")
    print(f"  SEP (exposicion)     0.3 * {ultimo['sep']:.3f}")
    print(f"  SVE (vulnerabilidad) 0.2 * {ultimo['sve']:.3f}")
    if pd.notna(ultimo.get("rendimiento_predicho")):
        print(f"Rendimiento predicho: {ultimo['rendimiento_predicho']:.2f} t/ha")
    if "is_anomaly" in ultimo and ultimo["is_anomaly"]:
        print("ALERTA: combinacion de variables anomala detectada")
    print("-" * 60)

    # Evolucion del IRA por cultivo
    piv = d.pivot_table(index="periodo", columns="cultivo", values="ira_score")
    if len(piv) > 1:
        piv.plot(marker="o", figsize=(10, 4),
                 title=f"Evolucion del IRA — {ultimo['nombre_municipio']}")
        plt.ylabel("IRA"); plt.tight_layout(); plt.show()

# Ejemplo: primer municipio con riesgo Critico disponible
critico = ira_last[ira_last["ira_nivel"] == "Crítico"]
CODIGO = str(critico["codigo_municipio"].iloc[0]) if len(critico) else str(ira_last["codigo_municipio"].iloc[0])
reporte_municipio(CODIGO)


---
### Resumen

- KPIs nacionales + distribución por nivel de riesgo.
- Ranking de municipios Alto/Crítico exportado a `notebooks/reportes/` (CSV y HTML).
- Gráficos parametrizables por departamento o cultivo.
- Reporte por municipio reutilizable — la base estática del reporte que en la plataforma
  enriquece el asistente LLM (`POST /api/municipio/{codigo}/chat`, `GET /reporte/{codigo}`).

Fin del flujo de notebooks. Para resultados en vivo, la plataforma corre con `make api` + `make web`.
